# 🤖 Simple RAG Demo — HR Policy Assistant

**RAG** stands for **Retrieval-Augmented Generation**. In plain English:

1. We take a document (here, an HR policy handbook).
2. We chop it into small pieces and turn each piece into a list of numbers (an "embedding") that captures its meaning.
3. We store those pieces in a searchable database (a "vector store").
4. When a user asks a question, we **search** the database for the most relevant pieces, and hand them to an AI model to **generate** a final answer.

That's it. No magic — just search + a language model.

**What we use in this notebook:**
- 📄 Data: `data/hr_policy.txt` (a sample HR policy document)
- 🔢 Embeddings: **Jina AI**
- 🗄️ Vector store: **FAISS**
- 🧠 LLM: **Groq** (fast + free-tier friendly)
- 🕸️ Framework: **LangChain** (`create_agent`)

> Before running: make sure the notebook's kernel is set to the `ragenv` virtual environment (top-right corner of VS Code / Jupyter), and that a `.env` file with `GROQ_API_KEY` and `JINA_API_KEY` exists in this folder.

## Step 1 — Import everything we need

We import all the tools upfront so it's clear what's being used and where it comes from.

In [40]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool

In [2]:
load_dotenv()

True

In [3]:
groq_key = os.getenv("GROQ_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")

print("ENVIORMENT VARIABLE LOADED")

ENVIORMENT VARIABLE LOADED


LOADING OUR DATA

In [4]:
DATA_FILE_PATH = os.path.join("Data","hr_policy.txt")

## DATA INGESTION

In [5]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()

print(documents)

[Document(metadata={'source': 'Data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probati

#### LANGCHAIN DOCUMENT 

Langchain processes everything in form of documents 


DOCUMENTS : 

PAGE CONTENT -- the actual data 

METADATA  - extra information about the data 

In [6]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [7]:
print(len(documents[0].page_content))
print(documents[0].metadata)

2598
{'source': 'Data\\hr_policy.txt'}



SPLITTING OUR DATA


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
    )

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'Data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'Data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'Data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [10]:
len(chunks)
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


EMBED OUR DATA

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


print("Embedding model ready and the name is",embeddings_model.model_parametrized_name)


Embedding model ready and the name is <bound method BaseModel.model_parametrized_name of <class 'langchain_google_genai.embeddings.GoogleGenerativeAIEmbeddings'>>


Store Data in Vector DB

In [14]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings_model)

print("chunks are stored",vector_store.index.ntotal)

chunks are stored 9


we never stored our data

In [15]:
test_query = "how many sick leaves employees get"

# similarity search


top_matches = vector_store.similarity_search(
    query=test_query,
    k=2
)
print(f"Query: {test_query}\n")

for i,match in enumerate(top_matches,start =1):
    print(f"---Match{i}---")
    print(match.page_content)
    print()


Query: how many sick leaves employees get

---Match1---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

---Match2---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



## TOOL CREATION

In [42]:
from langchain.tools import tool

retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

@tool
def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### DATA RETRIEVAL

Making LLM

In [ ]:
from langchain_groq import ChatGroq

llm =ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0  # it is like creativity of the model and ranges from 0 to 1
)

llm.model_name

'openai/gpt-oss-120b'

In [17]:
test_response = llm.invoke("Hey is learning RAG Hard? answer in one line")

print(test_response.content)

Learning RAG can be challenging at first, but with practice and good resources it becomes manageable.


AI AGENT

3 -- 

LLM - BRAIN 

TOOL - SUPER POWER 

MEMORY - no memory 

In [18]:
from langchain.agents import create_agent


In [37]:
# we created an agent hr assistant earlier we use llm which is a brain
hr_assistant = create_agent( 
    model = llm,
    tools=[search_hr_policy],
    system_prompt=""" 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [44]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [ ]:
ask_hr_assistant("what is the leaves policy")

QUESTION: what is the leaves policy
------------------------------------------------------------
ANSWER: **Acme Crop – Leave Policy Overview**

| Type of leave | Entitlement | Key details |
|---------------|------------|-------------|
| **Annual (paid) leave** | **20 days per calendar year** | • Submit requests through the HR portal **at least 5 working days** before the start date.<br>• Up to **5 unused days** can be carried forward to the next year; any excess is forfeited. |
| **Sick leave** | **10 paid days per year** | • A medical certificate is required for absences **longer than 2 consecutive days**.<br>• Sick leave is separate from annual leave. |
| **Unpaid leave (during probation)** | Not eligible for paid leave while on probation | • Employees may request **unpaid leave** for emergencies, subject to manager approval. |
| **Public holidays** | **12 days** per year (as per the official holiday calendar) | • If you must work on a public holiday, you’ll receive **compensatory le

'**Acme Crop – Leave Policy Overview**\n\n| Type of leave | Entitlement | Key details |\n|---------------|------------|-------------|\n| **Annual (paid) leave** | **20 days per calendar year** | • Submit requests through the HR portal **at least 5 working days** before the start date.<br>• Up to **5 unused days** can be carried forward to the next year; any excess is forfeited. |\n| **Sick leave** | **10 paid days per year** | • A medical certificate is required for absences **longer than 2 consecutive days**.<br>• Sick leave is separate from annual leave. |\n| **Unpaid leave (during probation)** | Not eligible for paid leave while on probation | • Employees may request **unpaid leave** for emergencies, subject to manager approval. |\n| **Public holidays** | **12 days** per year (as per the official holiday calendar) | • If you must work on a public holiday, you’ll receive **compensatory leave**. |\n\n**How to request leave**\n\n1. Log in to the **HR portal**.  \n2. Choose the leave ty

# what is response and other things 
# for example

In [ ]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me about leave policies"
            }
        ]
    }
)

In [39]:
response

{'messages': [HumanMessage(content='tell me about leave policiesr', additional_kwargs={}, response_metadata={}, id='6da3460e-d923-41f2-95c3-258aef9b11fb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer about leave policies. Must use search_hr_policy tool to look up facts before answering. So we need to query.', 'tool_calls': [{'id': 'fc_590ff91b-8dfb-4f6b-92dc-3f8398cb547a', 'function': {'arguments': '{"question":"leave policies"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 207, 'total_tokens': 264, 'completion_time': 0.119688323, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.009173179, 'prompt_tokens_details': None, 'queue_time': 0.318773528, 'total_time': 0.128861502}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_02b0d31eca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider'


SYSTEM MESSAGE - HR ASSISNT

HUMAN MESSAGE - TELL ME ABOUT POLICIES 

AI MESSAGE  - HEY THESE ARE THEPLOICES 



THE MESSAGES ARE IN THE LIST
FIRST WE HAVE SYSTEM MESSAGE
SECOND WE HAVE HUMAN MESSAGE
LAST WE HAVE AI MESSAGE

In [33]:
response["messages"][-1].content

"Sure! Here’s an overview of the most common leave policies at **Acme Corp**.\u202fIf you need details that are specific to your role, location, or tenure, please let me know or refer to the employee handbook/HR portal for the exact numbers.\n\n---\n\n### 1. Paid Time Off (PTO) / Vacation\n| **Eligibility** | **Accrual Rate** | **Maximum Balance** |\n|-----------------|------------------|---------------------|\n| All full‑time employees (after 90 days of service) | •\u202f0.83\u202fhours per pay period (≈10\u202fdays per year) for the first\u202f2\u202fyears  <br>•\u202f1.25\u202fhours per pay period (≈15\u202fdays per year) after 2\u202fyears  <br>•\u202f1.67\u202fhours per pay period (≈20\u202fdays per year) after 5\u202fyears | 30\u202fdays (any accrued days beyond this are capped; excess accrual is either paid out or forfeited per local law) |\n\n* **How to request:** Submit a request through the HR portal at least\u202f5\u202fbusiness days in advance for planned vacations. For sho

In [ ]:
print(response["messages"][-1].content) # try to save it in answer variable

**Acme Crop – Leave Policies (at a glance)**  

| Type of leave | Who’s eligible | How many days | How to request | Key notes |
|---------------|----------------|---------------|----------------|-----------|
| **Annual (paid) leave** | All full‑time employees | **20 days per calendar year** | Submit a request in the HR portal **at least 5 working days** before the start date | Up to **5 unused days** can be carried forward to the next year. Anything beyond that is forfeited. |
| **Sick leave** | All full‑time employees | **10 paid sick days per year** | Log the sick day in the HR portal; for absences **longer than 2 consecutive days**, attach a medical certificate. |
| **Unpaid leave (during probation)** | Employees in their **3‑month probation** period | Not applicable (no paid leave) | Request through the HR portal and obtain manager approval. Usually granted only for emergencies. |
| **Public holidays** | All employees | **12 days** (as per the annual holiday calendar) | No action n